# 01 — Run a Single Task

This notebook mirrors `pipelines/run_task.py`. It lets you run **one bio task** with a chosen framework and model, inspect the agent output, and see the evaluation score — all interactively.

### What you will learn
- How the `TASK_REGISTRY` and `FRAMEWORK_REGISTRY` work
- What a `BioTask` looks like (prompt, tools, evaluator)
- How an `AgentRunner` wraps a framework
- How results are scored with `EvalResult`

### Prerequisites
```bash
# From the repo root, activate the project environment:
uv sync --extra biomni --extra dev

# Make sure Ollama is running and llama3 is pulled:
ollama pull llama3
```

import os
import sys
import time
from pathlib import Path


def find_repo_root() -> Path:
    """Walk up from cwd until we find pyproject.toml (the repo root marker)."""
    current = Path(".").resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {current}")


repo_root = find_repo_root()
os.chdir(repo_root)

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Repo root:         {repo_root}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
import sys
import time
from pathlib import Path

# Ensure the src/ package is importable when running from notebooks/
repo_root = Path(".").resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Repo root: {repo_root}")

## 2. Explore available tasks and frameworks

Every task and framework is registered in a central registry. Let's see what's available.

In [ ]:
from bio_agents.frameworks import FRAMEWORK_REGISTRY
from bio_agents.tasks import TASK_REGISTRY

print("Available tasks:")
for name, cls in TASK_REGISTRY.items():
    task = cls()
    print(f"  {name:30s}  —  {task.description}")

print("\nAvailable frameworks:")
for name in FRAMEWORK_REGISTRY:
    print(f"  {name}")

## 3. Inspect a task

Each `BioTask` exposes:
- `get_input()` → the prompt sent to the agent
- `get_tools()` → any tool wrappers the agent can use
- `evaluate(result)` → scores the agent's output

In [ ]:
task_name = "literature_search"  # change me

task = TASK_REGISTRY[task_name]()
task_input = task.get_input()

print(f"Task:        {task.name}")
print(f"Description: {task.description}")
print(f"\n--- Prompt ---\n{task_input.prompt}")
print(f"\nContext: {task_input.context}")
print(f"\nTools available: {task.get_tools() or 'none (framework manages its own)'}")

## 4. Configure your run

Edit the three variables below to choose what to run.

In [ ]:
# --- Configuration ---
TASK = "literature_search"  # any key from TASK_REGISTRY above
FRAMEWORK = "biomni"  # any key from FRAMEWORK_REGISTRY above
MODEL = "llama3"  # local Ollama — free, no key needed
# alternatives:
#   "llama-3.3-70b"  → GROQ_API_KEY (free tier)
#   "claude-sonnet-4-6" → ANTHROPIC_API_KEY

# Biomni-specific kwargs (ignored by other frameworks)
RUNNER_KWARGS = {
    "use_tool_retriever": True,
    "timeout_seconds": 300,
    "commercial_mode": False,
}

print(f"Will run: task={TASK}  |  framework={FRAMEWORK}  |  model={MODEL}")

## 5. Run the task

Internally this creates a `BenchmarkConfig` with a single cell in the matrix and calls `BenchmarkRunner.run()`.

In [ ]:
from bio_agents.evaluation.runner import BenchmarkConfig, BenchmarkRunner

cfg = BenchmarkConfig(
    tasks=[TASK],
    frameworks=[FRAMEWORK],
    models=[MODEL],
    eval=["success_rate"],
    runner_kwargs=RUNNER_KWARGS,
)

runner = BenchmarkRunner(cfg)

print("Running... (this may take a minute)")
t0 = time.perf_counter()
results = runner.run()
elapsed = time.perf_counter() - t0

print(f"Done in {elapsed:.1f}s")

## 6. Inspect the result

In [ ]:
r = results[0]

print("=" * 60)
print(f"Task:      {r.task}")
print(f"Framework: {r.framework}")
print(f"Model:     {r.model}")
print(f"Duration:  {r.duration_s:.1f}s")
print("=" * 60)

print("\n--- Agent Output ---")
print(r.agent_result.output or "(no output)")

print("\n--- Evaluation ---")
print(f"Score:   {r.eval_result.score:.2f}")
print(f"Passed:  {r.eval_result.passed}")
print(f"Metrics: {r.eval_result.metrics}")

In [ ]:
# Optional: inspect intermediate tool calls / agent steps
if r.agent_result.tool_calls:
    print(f"{len(r.agent_result.tool_calls)} agent steps recorded:")
    for i, step in enumerate(r.agent_result.tool_calls[:5]):  # show first 5
        print(f"\n[Step {i}] {step.get('name', '')}")
        print(str(step.get("output", ""))[:300])
else:
    print("No tool calls recorded.")

## 7. Try a different task

Go back to **cell 4**, change `TASK` to one of the other registry keys (e.g. `gene_annotation`, `molecule_property`), and re-run from there.